# Spinal Disorder Classification — Stage 3: CNN Experiments

**Objective:** Using the permanently frozen Stage 2 split, compare the original-style CNN with an improved custom CNN and several image preprocessing strategies. We will use **only training and validation images** during this stage; the 51 held-out test images are not decoded, passed through a model, or used for selection.

**Prerequisite:** Your Stage 2 Colab ran successfully and created the private `splits/v1/` manifest and config. It reported 235 training, 52 validation, and 51 test images. The near-duplicate candidates were grouped, but **patient identity remains unverified**; results are not patient-independent.

**Privacy:** These X-rays and filenames may contain sensitive information. Keep private manifests, checkpoints, per-image results, and raw images out of public GitHub. This is an educational research prototype, **not a medical diagnostic device**.

**Use this as a separate Colab notebook.** Our Stages 1–2 notebook stays unchanged. Select **Runtime → Change runtime type → T4 GPU** if one is available.

## 1. Connect your dataset and freeze

Mount the Google Drive account containing your original `224/` image folder and Stage 2 frozen split. No re-splitting will occur.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports and experiment directory

All trained models and logs are saved privately under a new `experiments/stage3/` folder. We do **not** modify either the dataset or frozen split.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
EXP_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Experiment folder:', EXP_DIR)

## 3. Load the frozen split (and reject unexpected changes)

We read the test **metadata** only to confirm the partition exists; test images themselves are never loaded. The frozen dataset fingerprint must match the one recorded in Stage 2. This is an **image-level** evaluation: unknown patient overlap and source/acquisition differences remain limitations.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Shared, deterministic image preprocessing

We must treat **every class the same way**. Normalize raw pixels to `[0,1]` with a fixed factor (no dataset-wide fitted statistics). Never select preprocessing based on the held-out test images.

We compare:

- **RGB 128×128** (normalized reference)
- **Grayscale 128×128**
- **Grayscale + CLAHE 128×128**
- **Grayscale 224×224** (original source resolution)

The website will eventually use exactly the saved preprocessing configuration of the selected model.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def preprocess_image(path, mode, size):
    # Use EXIF orientation and identical operations for training, validation, and later inference.
    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(original, (size, size), method=Image.Resampling.LANCZOS, color=(0, 0, 0))
        rgb = np.asarray(resized, dtype=np.uint8)

    if mode == 'rgb':
        image = rgb
    elif mode in {'gray', 'clahe'}:
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        if mode == 'clahe':
            gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
        image = gray[..., np.newaxis]
    else:
        raise ValueError(f'Unknown preprocessing mode: {mode}')

    return image.astype(np.float32) / 255.0


def load_partition(rows, mode, size):
    X, y = [], []
    for record in rows.itertuples(index=False):
        source = DATA_ROOT / record.relative_path
        if not source.is_file() or file_sha256(source) != record.file_sha256:
            raise RuntimeError(f'Image missing or changed since freeze: {record.relative_path}')
        X.append(preprocess_image(source, mode, size))
        y.append(int(record.label))
    return np.stack(X), np.array(y, dtype=np.int64)

# Quick internal shape / range sanity check on a TRAINING image only.
sample = DATA_ROOT / train_df.iloc[0]['relative_path']
for mode in ('rgb', 'gray', 'clahe'):
    arr = preprocess_image(sample, mode=mode, size=128)
    print(mode, arr.shape, arr.dtype, (float(arr.min()), float(arr.max())))
    assert arr.shape == (128, 128, 3 if mode == 'rgb' else 1)
    assert np.isfinite(arr).all() and 0 <= arr.min() and arr.max() <= 1

## 5. Define the architectures

**Reference CNN:** resembles the original nine-layer network, now with normalization for stability. It is not an exact reconstruction of any old saved checkpoint.

**Custom CNN:** uses batch normalization, global average pooling, dropout, and mild regularization. This should substantially reduce dense-layer parameter count, but better validation performance is **not** guaranteed.

In [ ]:
from tensorflow.keras import layers, regularizers


def make_reference_cnn(shape):
    model = tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')
    return model


def make_custom_cnn(shape, seed):
    # Conservative geometric augmentation, TRAINING only. No horizontal flips.
    inputs = layers.Input(shape=shape)
    x = layers.RandomRotation(0.015, fill_mode='nearest', seed=seed)(inputs)
    x = layers.RandomTranslation(0.035, 0.035, fill_mode='nearest', seed=seed + 1)(x)
    x = layers.RandomZoom(0.035, fill_mode='nearest', seed=seed + 2)(x)

    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False,
                          kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.35, seed=seed + 3)(x)
    predictions = layers.Dense(3, activation='softmax')(x)
    return tf.keras.Model(inputs, predictions, name='custom_cnn')


def make_model(architecture, shape, seed):
    model = (make_reference_cnn(shape) if architecture == 'reference'
             else make_custom_cnn(shape, seed) if architecture == 'custom'
             else None)
    if model is None:
        raise ValueError(f'Unknown architecture: {architecture}')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

preview = make_model('custom', (128, 128, 1), SEED)
preview.summary()
print('Custom CNN parameters:', preview.count_params())
del preview
tf.keras.backend.clear_session()

## 6. Register our controlled experiments

The comparisons below are intentionally separated:

1. `E0` and `E1` compare **architecture** at fixed RGB preprocessing.
2. `E1`, `E2`, `E3` compare **RGB, grayscale, CLAHE** under the same custom CNN.
3. `E2`, `E4` compare **128 vs 224 pixels** under the same grayscale custom CNN.

**Start with `RUN_IDS = ['E0', 'E1']`.** When those finish, set it to `['E2', 'E3']`, and finally `['E4']`. Each run writes its own folder. Results are validation-only.

In [ ]:
EXPERIMENTS = {
    'E0': {'architecture': 'reference', 'mode': 'rgb', 'size': 128},
    'E1': {'architecture': 'custom', 'mode': 'rgb', 'size': 128},
    'E2': {'architecture': 'custom', 'mode': 'gray', 'size': 128},
    'E3': {'architecture': 'custom', 'mode': 'clahe', 'size': 128},
    'E4': {'architecture': 'custom', 'mode': 'gray', 'size': 224},
}

RUN_IDS = ['E0', 'E1']  # Later: ['E2', 'E3'] and then ['E4']
EPOCHS = 40
BATCH_SIZE = 16

assert all(run_id in EXPERIMENTS for run_id in RUN_IDS)
for run_id in RUN_IDS:
    assert not (EXP_DIR / run_id).exists(), (
        f'{run_id} already has an output folder. Do not overwrite previous experiments. '
        f'Choose an unrun experiment or investigate the existing directory.'
    )
print('Experiments to run:', RUN_IDS)

## 7. Train selected experiments

Training uses **235 training images**. Class weights are computed *only* from training labels. Validation drives early stopping, checkpoint selection, and model comparison; **the test set is never read**.

`ModelCheckpoint` writes the best model (lowest validation loss) to each experiment folder. A run directory is not overwritten automatically, even if the notebook is rerun.

In [ ]:
def train_experiment(experiment_id):
    details = dict(EXPERIMENTS[experiment_id])
    folder = EXP_DIR / experiment_id
    if folder.exists():
        raise FileExistsError(f'{folder} exists. Preserve past runs; do not overwrite.')

    # Load ONLY the frozen training and validation records.
    X_train, y_train = load_partition(train_df, details['mode'], details['size'])
    X_val, y_val = load_partition(val_df, details['mode'], details['size'])
    assert X_train.shape[0] == 235 and X_val.shape[0] == 52

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(details['architecture'], X_train.shape[1:], SEED)
    weights = compute_class_weight('balanced', classes=np.arange(3), y=y_train)
    class_weights = {int(k): float(v) for k, v in enumerate(weights)}

    metadata = {
        'experiment': experiment_id,
        **details,
        'seed': SEED,
        'epochs_max': EPOCHS,
        'batch_size': BATCH_SIZE,
        'optimizer': 'Adam', 'initial_learning_rate': 3e-4,
        'checkpoint_monitor': 'val_loss',
        'augmentation': 'custom CNN only: rotation, translation, zoom',
        'normalization': 'fixed pixel division by 255',
        'clahe_settings': {'clipLimit': 2.0, 'tileGridSize': [8, 8]} if details['mode'] == 'clahe' else None,
        'class_weights': class_weights,
        'dataset_fingerprint': fingerprint,
        'split_version': 'v1',
        'train_images': len(y_train), 'validation_images': len(y_val),
        'patient_independent': False,
        'test_set_evaluated': False,
        'parameters': int(model.count_params()),
    }

    folder.mkdir(parents=True, exist_ok=False)
    (folder / 'configuration.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    best_model_path = folder / 'best_model.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(best_model_path), monitor='val_loss', save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv')),
    ]
    print(f'\n=== {experiment_id}: {details} ===')
    t0 = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    # Make sure predictions come from the BEST saved validation-loss checkpoint.
    best_model = tf.keras.models.load_model(str(best_model_path))
    val_scores = best_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    val_predictions = np.argmax(val_scores, axis=1)
    metrics = {
        'experiment': experiment_id,
        'val_accuracy': float(accuracy_score(y_val, val_predictions)),
        'val_macro_f1': float(f1_score(y_val, val_predictions, labels=[0, 1, 2], average='macro', zero_division=0)),
        'val_balanced_accuracy': float(balanced_accuracy_score(y_val, val_predictions)),
        'best_val_loss': float(min(history.history['val_loss'])),
        'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
        'epochs_completed': len(history.history['loss']),
        'runtime_seconds': float(time.perf_counter() - t0),
        'per_class_recall': {},
        'test_evaluated': False,
    }
    report = classification_report(
        y_val, val_predictions, labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True, zero_division=0
    )
    for name in LABEL_TO_NAME.values():
        metrics['per_class_recall'][name] = float(report[name]['recall'])

    (folder / 'validation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    (folder / 'classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    pd.DataFrame(history.history).to_csv(folder / 'history.csv', index=False)
    cm = confusion_matrix(y_val, val_predictions, labels=[0, 1, 2])
    np.save(folder / 'confusion_matrix_validation.npy', cm)
    # Per-image predictions/filenames are deliberately NOT included in the public summary.

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, axis in [('loss', axes[0]), ('accuracy', axes[1])]:
        axis.plot(history.history[name], label='train')
        axis.plot(history.history[f'val_{name}'], label='validation')
        axis.set_title(f'{experiment_id} {name}')
        axis.set_xlabel('Epoch')
        axis.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

    print('Validation confusion matrix: rows=true, columns=predicted')
    display(pd.DataFrame(cm, index=list(LABEL_TO_NAME.values()), columns=list(LABEL_TO_NAME.values())))
    print('VALIDATION metrics only:', json.dumps(metrics, indent=2))
    return metrics

run_results = []
for experiment_id in RUN_IDS:
    run_results.append(train_experiment(experiment_id))

display(pd.DataFrame(run_results).drop(columns=['per_class_recall']).round(4))

## 8. Compare all previously completed runs (validation data only)

This reads summary JSON files from finished runs. Use validation **macro F1 as the primary model-selection metric**, with validation loss and per-class recall as important secondary checks. Select a final model only after completing all planned experiments; **do not use final test results to select it**.

In [ ]:
completed = []
for exp_id in EXPERIMENTS:
    metrics_path = EXP_DIR / exp_id / 'validation_metrics.json'
    if metrics_path.is_file():
        data = json.loads(metrics_path.read_text(encoding='utf-8'))
        assert data['test_evaluated'] is False
        completed.append(data)

if completed:
    leaderboard = pd.DataFrame(completed)
    leaderboard = leaderboard.sort_values(
        ['val_macro_f1', 'best_val_loss'], ascending=[False, True]
    )
    display(leaderboard.drop(columns=['per_class_recall']).round(4))
    print('Current validation-only leader:', leaderboard.iloc[0]['experiment'])
else:
    print('No completed runs yet. Execute Section 7 first.')

## Stage 3 checkpoint

Before Stage 4 / final model selection:

- Complete E0–E4, preserving each model checkpoint and configuration.
- Compare **macro F1**, per-class recall, confusion matrices, validation accuracy, and learning curves. A high accuracy that ignores Normal or Spondylolisthesis is not sufficient.
- Remember the validation set contains **52** images: one changed prediction shifts accuracy by about **1.92 percentage points**. Treat small improvements cautiously.
- If results are unstable, use repeated training seeds or development-set cross-validation for robustness; don't move or peek at the test split to make results look better.
- **Only after model selection is locked:** load the chosen checkpoint, run the final evaluation on the 51 held-out images once, and explicitly report the limitations of unverified patient independence and potential acquisition/source shortcuts.
- Keep the full notebook and private logs in Drive; publish cleaned source code, reproducible configuration, aggregate results, and scrubbed notebook outputs to GitHub.

**Send the E0/E1 training outputs or Stage 3 validation leaderboard back into this conversation.** We'll use those results to decide which experiment deserves further optimization.